# Electrostatic potential (Poisson equation)

*em · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/em/electrostatics)

Solve Poisson's equation for the potential of a few line charges inside a grounded box. On a grid the Laplacian $\partial_i\partial_i$ becomes a matrix built from the 1D second-difference matrix with Kronecker products, so the whole PDE turns into one linear system $L\phi = -\rho/\varepsilon_0$ that torch.linalg.solve handles in a single call.

**Vector form**

$$\nabla^2\phi = -\rho/\varepsilon_0,\qquad \mathbf E = -\nabla\phi$$

**Index (tensor) form**

$$\partial_i\partial_i\,\phi = -\rho/\varepsilon_0 \;\;\longrightarrow\;\; L_{nm}\,\phi_m = -\rho_n/\varepsilon_0,\qquad E_i = -\partial_i\phi$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\phi_{ab}$ | `phi` | (N, N) | potential at grid row $a$ ($y$) and column $b$ ($x$) |
| $\phi_m$ | `phi.reshape(-1)` | (N²,) | same values flattened: $m = a\cdot N + b$ |
| $D_{ab}$ | `second_difference(N, h)` | (N, N) | 1D $\partial^2$: $(\phi_{a-1} - 2\phi_a + \phi_{a+1})/h^2$, tridiagonal |
| $L_{nm}$ | `kron(D, I) + kron(I, D)` | (N², N²) | 2D Laplacian $\partial_y^2 + \partial_x^2$: banded, five non-zeros per row |
| $\rho_n$ | `rho.reshape(-1)` | (N²,) | charge density on the grid |
| $L_{nm}\phi_m$ | `einsum('nm,m->n', L, phi)` | (N²,) | repeated $m$ ⇒ summed: the discrete Laplacian of $\phi$ |
| $E_i$ | `electric_field(phi, h)` | (2, N, N) | $E_i = -\partial_i\phi$ by central differences, $i \in \{x, y\}$ |

The box is the unit square with $\phi = 0$ on its walls (grounded). The problem is 2D, so each "point" charge is really an infinite line charge $\lambda$ (per unit length) and Gauss's law reads $\oint \mathbf E\cdot\mathbf n\,dl = \lambda/\varepsilon_0$. Units: $\varepsilon_0 = 1$. $L$ is mostly zeros — a real code would store it as a sparse matrix; here $N^2\times N^2$ dense is still small enough to solve directly.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/em/electrostatics/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli

from . import plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    n = params["n_grid"]
    x, h = tensors.grid(n)
    rho = tensors.charge_density(params["config"], n, params["charge"])  # CPU float64: a direct solve

    phi, E = solve.solve(rho, h)

    print(f"grid {n}x{n}: {n * n} unknowns, potential range [{phi.min():.3f}, {phi.max():.3f}]")
    return plot.draw(x, phi, E, rho, out_dir / "electrostatics.png")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/em/electrostatics/equation.py
r"""Physics → tensor translation.

Gauss's law with E = −∇φ gives Poisson's equation for the potential,

    \nabla^2 \phi = -\rho / \varepsilon_0,

in index notation (i is repeated, so it is summed over x and y):

    \partial_i \partial_i \phi = -\rho / \varepsilon_0,      E_i = -\partial_i \phi

On an N × N grid with spacing h the 1D second derivative is a tridiagonal matrix

    D_{ab} φ_b = (φ_{a-1} - 2 φ_a + φ_{a+1}) / h²

Flatten the grid φ_{ab} (row a = y, column b = x) into one vector φ_m with m = a·N + b.
Then ∂_y² acts on the row index and ∂_x² on the column index, which is a Kronecker product:

    L = D ⊗ I + I ⊗ D,        L_{nm} φ_m = -ρ_n / ε_0

The repeated m is summed: one big matrix–vector product, solved for φ in one call.
"""

import torch


def second_difference(n: int, h: float, dtype=torch.float64) -> torch.Tensor:
    """D of shape (N, N): -2 on the diagonal, 1 on both neighbours, divided by h².

    Missing neighbours at the ends are the grounded walls (φ = 0), so they simply drop out.
    """
    main = -2.0 * torch.ones(n, dtype=dtype)
    side = torch.ones(n - 1, dtype=dtype)
    return (torch.diag(main) + torch.diag(side, 1) + torch.diag(side, -1)) / h**2


def laplacian(n: int, h: float) -> torch.Tensor:
    """L = D ⊗ I + I ⊗ D of shape (N², N²): ∂_y² on rows plus ∂_x² on columns."""
    D = second_difference(n, h)
    I = torch.eye(n, dtype=D.dtype)  # noqa: E741
    return torch.kron(D, I) + torch.kron(I, D)


def apply_laplacian(L: torch.Tensor, phi: torch.Tensor) -> torch.Tensor:
    r"""(∂_i∂_i φ)_n = L_{nm} φ_m.   phi: (N, N) → (N, N)"""
    n = phi.shape[0]
    return torch.einsum("nm,m->n", L, phi.reshape(-1)).reshape(n, n)


def electric_field(phi: torch.Tensor, h: float) -> torch.Tensor:
    """E_i = -∂_i φ with central differences; the grounded walls (φ = 0) are padded on.  → (2, N, N): (E_x, E_y)"""
    padded = torch.nn.functional.pad(phi, (1, 1, 1, 1))  # (N+2, N+2) walls at φ = 0
    E_x = -(padded[1:-1, 2:] - padded[1:-1, :-2]) / (2 * h)  # column index b is x
    E_y = -(padded[2:, 1:-1] - padded[:-2, 1:-1]) / (2 * h)  # row index a is y
    return torch.stack([E_x, E_y])


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/em/electrostatics/tensors.py
"""Build the concrete tensors: the grid and the charge density ρ for each layout."""

import torch

# Line charges as (x, y, sign) in the unit square; "capacitor" uses two charged plates instead.
LAYOUTS = {
    "dipole": [(0.35, 0.5, +1), (0.65, 0.5, -1)],
    "quadrupole": [(0.35, 0.35, +1), (0.65, 0.65, +1), (0.35, 0.65, -1), (0.65, 0.35, -1)],
    "same_sign": [(0.35, 0.5, +1), (0.65, 0.5, +1)],
}


def grid(n: int) -> tuple[torch.Tensor, float]:
    """Interior grid coordinates x_b = (b + 1) h of shape (N,), spacing h = 1 / (N + 1)."""
    h = 1.0 / (n + 1)
    return torch.arange(1, n + 1, dtype=torch.float64) * h, h


def charge_density(config: str, n: int, charge: float) -> torch.Tensor:
    """ρ_{ab} of shape (N, N) (row a = y, column b = x).

    A line charge λ sits in a single cell, so its density is λ / h² (the cell area is h²).
    """
    _, h = grid(n)
    rho = torch.zeros(n, n, dtype=torch.float64)
    if config == "capacitor":
        lo, hi = round(0.3 * n), round(0.7 * n)  # plates span 0.3 < y < 0.7
        per_cell = charge / (hi - lo)  # total charge λ on each plate
        rho[lo:hi, round(0.35 * n)] = +per_cell / h**2
        rho[lo:hi, round(0.65 * n)] = -per_cell / h**2
        return rho
    for x, y, sign in LAYOUTS[config]:
        a, b = round(y / h) - 1, round(x / h) - 1  # nearest interior cell
        rho[a, b] += sign * charge / h**2
    return rho


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/em/electrostatics/solve.py
"""Solve L φ = −ρ/ε₀ for the potential, then take E = −∇φ."""

import torch

from .equation import electric_field, laplacian


def solve(rho: torch.Tensor, h: float, eps0: float = 1.0) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (φ of shape (N, N), E of shape (2, N, N)).

    The whole PDE is one dense linear system of size N² — fine for N ≲ 80 (6400 unknowns).
    """
    n = rho.shape[0]
    L = laplacian(n, h)  # (N², N²)
    phi = torch.linalg.solve(L, -rho.reshape(-1) / eps0).reshape(n, n)  # (N, N)
    return phi, electric_field(phi, h)


def flux_through_square(E: torch.Tensor, h: float, a0: int, a1: int, b0: int, b1: int) -> torch.Tensor:
    """∮ E·n dl around the box of rows a0..a1 and columns b0..b1 (outward normal n), by the midpoint rule."""
    E_x, E_y = E
    right = E_x[a0 : a1 + 1, b1].sum()
    left = -E_x[a0 : a1 + 1, b0].sum()
    top = E_y[a1, b0 : b1 + 1].sum()
    bottom = -E_y[a0, b0 : b1 + 1].sum()
    return (right + left + top + bottom) * h


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/em/electrostatics/plot.py
"""Static figure: potential heatmap with equipotentials, field streamlines and the charges."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import save_png


def draw(x: torch.Tensor, phi: torch.Tensor, E: torch.Tensor, rho: torch.Tensor, path: Path) -> Path:
    xs = x.numpy()  # (N,) same coordinates for x and y
    p = phi.numpy()  # (N, N) rows = y, columns = x
    E_x, E_y = E.numpy()
    limit = np.quantile(np.abs(p), 0.98)  # φ ~ log r blows up at a line charge: clip the colour scale

    fig, ax = plt.subplots(figsize=(6, 5))
    image = ax.imshow(
        p, origin="lower", extent=(0, 1, 0, 1), cmap="RdBu_r", vmin=-limit, vmax=limit, interpolation="bilinear"
    )
    ax.contour(xs, xs, p, levels=np.linspace(-limit, limit, 13), colors="k", linewidths=0.6, alpha=0.6)
    ax.streamplot(xs, xs, E_x, E_y, color="#374151", density=1.1, linewidth=0.7, arrowsize=0.8)

    a, b = np.nonzero(rho.numpy())  # cells that carry charge
    sign = np.sign(rho.numpy()[a, b])
    ax.scatter(
        xs[b], xs[a], c=np.where(sign > 0, "#dc2626", "#2563eb"), s=max(8, 60 / len(a)), edgecolors="k", zorder=3
    )

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title("potential φ, equipotentials and field lines E = −∇φ")
    ax.grid(False)
    fig.colorbar(image, ax=ax, label="φ")
    fig.tight_layout()
    return save_png(fig, path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'config': 'dipole',
    'n_grid': 60,
    'charge': 1.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.em.electrostatics")]:
    del sys.modules[name]
main = importlib.import_module("problems.em.electrostatics.main")
problem_dir = Path("problems/em/electrostatics")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
